In [1]:
import pandas as pd
from pathlib import Path
from typing import List
import xml.etree.ElementTree as ET

In [4]:
from pathlib import Path

all_xml = list(Path('./source/10').rglob("*.xml"))
properties_xml = [p for p in all_xml if p.name.endswith('_Properties.xml')]

print("Properties files only:")
for p in sorted(properties_xml):
    print(f"  - {p.name}")

Properties files only:
  - 10 P 1_Properties.xml
  - 10 P 2_Properties.xml
  - 10 P 3_Properties.xml
  - 10 P 4_Properties.xml
  - 10 P 5_Properties.xml


In [7]:
import pandas as pd
from pathlib import Path
import xml.etree.ElementTree as ET
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font
from openpyxl.utils import get_column_letter

# ================================
# Configuration
# ================================
source_folder = Path("./source/10")
output_excel  = Path("Leica_Channels_Comparison_Detailed.xlsx")

prop_files = sorted(source_folder.rglob("*_Properties.xml"))
print(f"Found {len(prop_files)} Properties XML files\n")

channel0_data = []
channel1_data = []

for xml_path in prop_files:
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        file_info = {
            "File": xml_path.name,
            "Folder": str(xml_path.relative_to(source_folder.parent))
        }

        # Find channel blocks by UserDefName (or fallback to order)
        channels = root.findall(".//WideFieldChannelInfo")
        ch_bf   = None  # Brightfield
        ch_fluo = None  # Fluorescence

        for ch in channels:
            name = ch.get("UserDefName", "")
            if "New 1" in name or "BF" in name.upper():
                ch_bf = ch
            elif "New 2" in name or "FLUOR" in name.upper() or "AutoFocus" not in name:
                ch_fluo = ch

        # Fallback: assume first = BF, second = FLUO
        if not ch_bf and len(channels) >= 1:   ch_bf   = channels[0]
        if not ch_fluo and len(channels) >= 2: ch_fluo = channels[1]

        def extract(ch, label):
            if ch is None:
                return {**file_info, "Channel": label, "Status": "Missing"}

            # Common dimension values (same for all channels)
            dim_x = root.find('.//DimensionDescription[@DimID="X"]')
            voxel = dim_x.get("Voxel", "0.11") if dim_x is not None else "0.11"

            return {
                **file_info,
                "Channel": label,
                "Contrast": ch.get("ContrastingMethodName", ""),
                "Bit depth": "12-bit",
                "Scale (µm/pixel)": voxel,
                "Image size": "1200×1200 px",
                "Physical size": "131.39 µm",
                "Black point": ch.get("BlackValue", ""),
                "White point": ch.get("WhiteValue", ""),
                "Gamma": ch.get("GammaValue", "1"),
                "Light intensity": ch.get("TL_Light-Intensity", "") or ch.get("IL_Light-Intensity", ""),
                "LED Intensity": ch.get("ILLEDIntensity4", "") or ch.get("ILLEDDiscreteIntensity4", ""),
                "Exposure time": ch.get("ExposureTime", ""),
                "Gain": ch.get("StringGainName", ""),
                "LUT": ch.get("LUT", ""),
                "Emission Filter": ch.get("FFW_Emission1FilterName", ""),
            }

        channel0_data.append(extract(ch_bf,   "Channel 0 (Brightfield)"))
        channel1_data.append(extract(ch_fluo, "Channel 1 (Fluorescence)"))

        print(f"Success: Processed {xml_path.name}")

    except Exception as e:
        print(f"Failed: Error in {xml_path.name}: {e}")
        channel0_data.append({**file_info, "Status": f"Error: {e}"})
        channel1_data.append({**file_info, "Status": f"Error: {e}"})

# ================================
# Write to Excel
# ================================
with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
    pd.DataFrame(channel0_data).to_excel(writer, sheet_name="Channel 0 - Brightfield",   index=False)
    pd.DataFrame(channel1_data).to_excel(writer, sheet_name="Channel 1 - Fluorescence", index=False)

# ================================
# Apply highlighting + safe column width
# ================================
wb = load_workbook(output_excel)
green  = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
orange = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")
header_fill = PatternFill(start_color="366092", end_color="366092", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]

    # Header style
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font

    # Identify value columns (skip File/Folder/Channel)
    value_cols = []
    for col_idx, cell in enumerate(ws[1], start=1):
        if cell.value not in ["File", "Folder", "Channel", "Status"]:
            value_cols.append(col_idx)

    # Row-by-row difference highlighting
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        values = [row[col-1].value for col in value_cols]
        if not values or all(v is None for v in values):
            continue

        first = str(values[0]).strip()
        all_same = all(str(v).strip() == first for v in values if v is not None)

        fill = green if all_same else orange
        for col_idx in value_cols:
            row[col_idx-1].fill = fill

    # Safe column auto-width (skips MergedCell)
    for col_idx in range(1, ws.max_column + 1):
        max_length = 0
        column = get_column_letter(col_idx)
        for cell in ws[column]:
            if not hasattr(cell, "value") or cell.value is None:
                continue
            try:
                max_length = max(max_length, len(str(cell.value)))
            except:
                pass
        adjusted_width = min(max_length + 2, 60)
        ws.column_dimensions[column].width = adjusted_width

wb.save(output_excel)

print(f"\nDone! Excel with highlighted differences saved:")
print(f"   {output_excel.resolve()}")
print(f"\nSheets created:")
print(f"   • Channel 0 - Brightfield")
print(f"   • Channel 1 - Fluorescence")
print(f"\nGreen  = identical values")
print(f"Orange = different values")

Found 5 Properties XML files

Success: Processed 10 P 1_Properties.xml
Success: Processed 10 P 2_Properties.xml
Success: Processed 10 P 3_Properties.xml
Success: Processed 10 P 4_Properties.xml
Success: Processed 10 P 5_Properties.xml

Done! Excel with highlighted differences saved:
   C:\Users\hochi\dev\pd\Leica_Channels_Comparison_Detailed.xlsx

Sheets created:
   • Channel 0 - Brightfield
   • Channel 1 - Fluorescence

Green  = identical values
Orange = different values


C:\Users\hochi\AppData\Local\Temp\ipykernel_15032\774256257.py:43: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  if not ch_bf and len(channels) >= 1:   ch_bf   = channels[0]
C:\Users\hochi\AppData\Local\Temp\ipykernel_15032\774256257.py:44: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  if not ch_fluo and len(channels) >= 2: ch_fluo = channels[1]
